<a href="https://colab.research.google.com/github/dsatish1252/sqlite-python-implementation/blob/main/Hospital_OPD_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAP HANA Cloud Hospital Management Analytics

## Setup and Connection

This section handles the installation of the necessary library and establishes the connection to the SAP HANA Cloud instance.

In [1]:
!pip install hdbcli

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 68.2 MB/s eta 0:00:00


### Connect to SAP HANA Cloud

Establishes a connection to the SAP HANA Cloud database using the provided credentials. **Note:** `sslValidateCertificate=False` is used for demonstration purposes. In a production environment, proper SSL certificate validation should always be enabled.

In [8]:
from hdbcli import dbapi

HANA_HOST = "13b7c15d-848f-40b5-9259-c9c36ab85f56.hna1.prod-eu10.hanacloud.ondemand.com"
HANA_PORT = 443
HANA_USER = "GE337044"
HANA_PASSWORD = "ObOukfvjH61!"

conn = dbapi.connect(
    address=HANA_HOST,
    port=HANA_PORT,
    user=HANA_USER,
    password=HANA_PASSWORD,
    encrypt=True,
    sslValidateCertificate=False  # For training/demo. Use certificate validation in production.
)

cursor = conn.cursor()

print("Connected successfully to SAP HANA Cloud")

Connected successfully to SAP HANA Cloud


### Verify Connection

Checks the current user and schema to confirm a successful connection.

In [9]:
cursor.execute("SELECT CURRENT_USER, CURRENT_SCHEMA FROM DUMMY")
result = cursor.fetchone()

print("Current User:", result[0])
print("Current Schema:", result[1])

Current User: GE337044
Current Schema: GE337044


## Data Preparation: Drop Existing Tables

This step ensures a clean slate by dropping any existing tables that might interfere with the creation of new tables. Errors are caught for tables that may not exist.

In [10]:
tables_to_drop = ["PATIENTS", "DEPARTMENTS", "DOCTORS", "APPOINTMENTS","BILLING"]

for table in tables_to_drop:
    try:
        cursor.execute(f"DROP TABLE {table}")
        print(f"Dropped table: {table}")
    except Exception as e:
        print(f"Table {table} does not exist or cannot be dropped. Skipping.")

conn.commit()

Table PATIENTS does not exist or cannot be dropped. Skipping.
Table DEPARTMENTS does not exist or cannot be dropped. Skipping.
Table DOCTORS does not exist or cannot be dropped. Skipping.
Table APPOINTMENTS does not exist or cannot be dropped. Skipping.
Table BILLING does not exist or cannot be dropped. Skipping.


## Table Creation

This section defines and creates the schema for the Hospital Management System, including tables for Patients, Departments, Doctors, Appointments, and Billing.

### Create `PATIENTS` Table

In [11]:
cursor.execute("""

CREATE COLUMN TABLE PATIENTS(

PATIENT_ID INTEGER PRIMARY KEY,

PATIENT_NAME NVARCHAR(100),

GENDER NVARCHAR(10),

AGE INTEGER,

CITY NVARCHAR(50),

MOBILE_NUMBER NVARCHAR(15)

)

""")

True

### Create `DEPARTMENTS` Table

In [12]:
cursor.execute("""

CREATE COLUMN TABLE DEPARTMENTS(

DEPARTMENT_ID INTEGER PRIMARY KEY,

DEPARTMENT_NAME NVARCHAR(100),

FLOOR_NUMBER INTEGER

)

""")

True

### Create `DOCTORS` Table

Includes a foreign key reference to the `DEPARTMENTS` table.

In [13]:
cursor.execute("""

CREATE COLUMN TABLE DOCTORS(

DOCTOR_ID INTEGER PRIMARY KEY,

DOCTOR_NAME NVARCHAR(100),

DEPARTMENT_ID INTEGER,

SPECIALIZATION NVARCHAR(100),

CONSULTATION_FEE DECIMAL(10,2),

FOREIGN KEY(DEPARTMENT_ID)

REFERENCES DEPARTMENTS(DEPARTMENT_ID)

)

""")

True

### Create `APPOINTMENTS` Table

Includes foreign key references to `PATIENTS` and `DOCTORS` tables.

In [14]:
cursor.execute("""

CREATE COLUMN TABLE APPOINTMENTS(

APPOINTMENT_ID INTEGER PRIMARY KEY,

PATIENT_ID INTEGER,

DOCTOR_ID INTEGER,

APPOINTMENT_DATE DATE,

APPOINTMENT_TIME TIME,

APPOINTMENT_STATUS NVARCHAR(20),

FOREIGN KEY(PATIENT_ID)

REFERENCES PATIENTS(PATIENT_ID),

FOREIGN KEY(DOCTOR_ID)

REFERENCES DOCTORS(DOCTOR_ID)

)

""")

True

### Create `BILLING` Table

Includes a foreign key reference to the `APPOINTMENTS` table with a unique constraint on `APPOINTMENT_ID`.

In [15]:
cursor.execute("""

CREATE COLUMN TABLE BILLING(

BILL_ID INTEGER PRIMARY KEY,

APPOINTMENT_ID INTEGER UNIQUE,

BILL_AMOUNT DECIMAL(10,2),

PAYMENT_MODE NVARCHAR(20),

PAYMENT_STATUS NVARCHAR(20),

FOREIGN KEY(APPOINTMENT_ID)

REFERENCES APPOINTMENTS(APPOINTMENT_ID)

)

""")

True

## Data Insertion

This section populates the newly created tables with sample data for demonstration and analysis.

### Insert Data into `DEPARTMENTS` Table

In [16]:
cursor.executemany("""

INSERT INTO DEPARTMENTS VALUES (?,?,?)

""", [

(1,'Cardiology',2),

(2,'Orthopedics',3),

(3,'General Medicine',1),

(4,'Pediatrics',2)

])

(1, 1, 1, 1)

In [43]:
cursor.executemany("""
INSERT INTO DEPARTMENTS VALUES (?,?,?)
""",[
(5,'Neurology',4),
(6,'Dermatology',1)
])

conn.commit()

### Insert Data into `PATIENTS` Table

In [17]:
cursor.executemany("""

INSERT INTO PATIENTS VALUES (?,?,?,?,?,?)

""",[

(101,'Rahul','Male',35,'Hyderabad','9876543210'),

(102,'Sneha','Female',29,'Vijayawada','9876543211'),

(103,'Amit','Male',42,'Chennai','9876543212'),

(104,'Priya','Female',31,'Bangalore','9876543213'),

(105,'Kiran','Male',24,'Hyderabad','9876543214')

])

(1, 1, 1, 1, 1)

In [44]:
cursor.executemany("""
INSERT INTO PATIENTS VALUES (?,?,?,?,?,?)
""",[
(106,'Anjali Verma','Female',38,'Mumbai','9876543215'),
(107,'Rakesh Patel','Male',45,'Pune','9876543216'),
(108,'Meena Gupta','Female',27,'Delhi','9876543217'),
(109,'Suresh Yadav','Male',52,'Visakhapatnam','9876543218'),
(110,'Pooja Nair','Female',34,'Kochi','9876543219'),
(111,'Arjun Das','Male',30,'Bhubaneswar','9876543220'),
(112,'Divya Iyer','Female',26,'Chennai','9876543221')
])

conn.commit()

### Insert Data into `DOCTORS` Table

In [18]:
cursor.executemany("""

INSERT INTO DOCTORS VALUES (?,?,?,?,?)

""",[

(1,'Dr. Sharma',1,'Cardiologist',800),

(2,'Dr. Reddy',2,'Orthopedic',700),

(3,'Dr. Mehta',3,'Physician',500),

(4,'Dr. Rao',4,'Pediatrician',600)

])

(1, 1, 1, 1)

In [45]:
cursor.executemany("""
INSERT INTO DOCTORS VALUES (?,?,?,?,?)
""",[
(5,'Dr. Gupta',5,'Neurologist',1000),
(6,'Dr. Thomas',6,'Dermatologist',650),
(7,'Dr. Kapoor',1,'Heart Specialist',900),
(8,'Dr. Srinivas',3,'General Physician',550)
])

conn.commit()

### Insert Data into `APPOINTMENTS` Table

In [19]:
cursor.executemany("""

INSERT INTO APPOINTMENTS VALUES (?,?,?,?,?,?)

""",[

(1001,101,1,'2026-07-01','10:00:00','Completed'),

(1002,102,2,'2026-07-01','11:00:00','Completed'),

(1003,103,3,'2026-07-02','09:30:00','Pending'),

(1004,104,1,'2026-07-02','12:00:00','Cancelled'),

(1005,105,4,'2026-07-02','01:00:00','Completed')

])

(1, 1, 1, 1, 1)

In [46]:
cursor.executemany("""
INSERT INTO APPOINTMENTS VALUES (?,?,?,?,?,?)
""",[
(1006,106,5,'2026-07-03','09:00:00','Completed'),
(1007,107,6,'2026-07-03','10:15:00','Completed'),
(1008,108,7,'2026-07-03','11:30:00','Pending'),
(1009,109,8,'2026-07-04','09:45:00','Completed'),
(1010,110,2,'2026-07-04','10:30:00','Cancelled'),
(1011,111,3,'2026-07-04','11:00:00','Completed'),
(1012,112,4,'2026-07-05','12:30:00','Completed'),
(1013,101,5,'2026-07-05','13:00:00','Pending'),
(1014,102,6,'2026-07-05','14:00:00','Completed'),
(1015,103,7,'2026-07-06','09:30:00','Completed')
])

conn.commit()

### Insert Data into `BILLING` Table and Commit

Commits all pending transactions to the database.

In [20]:
cursor.executemany("""

INSERT INTO BILLING VALUES (?,?,?,?,?)

""",[

(1,1001,800,'UPI','Paid'),

(2,1002,700,'Cash','Paid'),

(3,1003,500,'Card','Unpaid'),

(4,1004,800,'Insurance','Refunded'),

(5,1005,600,'UPI','Paid')

])

conn.commit()

In [47]:
cursor.executemany("""
INSERT INTO BILLING VALUES (?,?,?,?,?)
""",[
(6,1006,1000,'Card','Paid'),
(7,1007,650,'Cash','Paid'),
(8,1008,900,'UPI','Unpaid'),
(9,1009,550,'Card','Paid'),
(10,1010,700,'Insurance','Refunded'),
(11,1011,500,'Cash','Paid'),
(12,1012,600,'UPI','Paid'),
(13,1013,1000,'Card','Unpaid'),
(14,1014,650,'Cash','Paid'),
(15,1015,900,'UPI','Paid')
])

conn.commit()

## Data Verification: Display All Tables

Retrieves and displays the content of all created tables to verify data insertion and table structure. Output is formatted as pandas DataFrames.

In [48]:
import pandas as pd

tables = [
    "PATIENTS",
    "DEPARTMENTS",
    "DOCTORS",
    "APPOINTMENTS",
    "BILLING"
]

for table in tables:
    print(f"\n----- {table} -----")
    cursor.execute(f"SELECT * FROM {table}")
    rows = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(rows, columns=columns)
    print(df)


----- PATIENTS -----
    PATIENT_ID  PATIENT_NAME  GENDER  AGE           CITY MOBILE_NUMBER
0          101         Rahul    Male   35      Hyderabad    9876543210
1          102         Sneha  Female   29     Vijayawada    9876543211
2          103          Amit    Male   42        Chennai    9876543212
3          104         Priya  Female   31      Bangalore    9876543213
4          105         Kiran    Male   24      Hyderabad    9876543214
5          106  Anjali Verma  Female   38         Mumbai    9876543215
6          107  Rakesh Patel    Male   45           Pune    9876543216
7          108   Meena Gupta  Female   27          Delhi    9876543217
8          109  Suresh Yadav    Male   52  Visakhapatnam    9876543218
9          110    Pooja Nair  Female   34          Kochi    9876543219
10         111     Arjun Das    Male   30    Bhubaneswar    9876543220
11         112    Divya Iyer  Female   26        Chennai    9876543221

----- DEPARTMENTS -----
   DEPARTMENT_ID   DEPARTMENT_

## View Creation

This section creates SQL views to simplify complex queries and provide pre-joined datasets for reporting and analytics.

In [49]:
cursor.execute("""

CREATE VIEW V_OPD_APPOINTMENT_ANALYTICS AS

SELECT

A.APPOINTMENT_ID,

A.APPOINTMENT_DATE,

A.APPOINTMENT_TIME,

P.PATIENT_NAME,

P.CITY AS PATIENT_CITY,

D.DOCTOR_NAME,

DEP.DEPARTMENT_NAME,

D.SPECIALIZATION,

D.CONSULTATION_FEE,

B.BILL_AMOUNT,

B.PAYMENT_MODE,

B.PAYMENT_STATUS,

A.APPOINTMENT_STATUS

FROM APPOINTMENTS A

JOIN PATIENTS P

ON A.PATIENT_ID=P.PATIENT_ID

JOIN DOCTORS D

ON A.DOCTOR_ID=D.DOCTOR_ID

JOIN DEPARTMENTS DEP

ON D.DEPARTMENT_ID=DEP.DEPARTMENT_ID

JOIN BILLING B

ON A.APPOINTMENT_ID=B.APPOINTMENT_ID

""")

ProgrammingError: (322, 'cannot use duplicate view name: V_OPD_APPOINTMENT_ANALYTICS: line 3 col 13 (at pos 14)')

## Analytics and Reporting Queries

This section contains various analytical queries to extract insights from the hospital management data.

### Find all completed OPD appointments



In [50]:
import pandas as pd

query = """
SELECT
    APPOINTMENT_ID,
    PATIENT_NAME,
    DOCTOR_NAME,
    DEPARTMENT_NAME,
    APPOINTMENT_DATE,
    BILL_AMOUNT,
    PAYMENT_STATUS
FROM V_OPD_APPOINTMENT_ANALYTICS
WHERE APPOINTMENT_STATUS = 'Completed';
"""

cursor.execute(query)

rows = cursor.fetchall()
columns = [col[0] for col in cursor.description]
df_completed_appointments = pd.DataFrame(rows, columns=columns)

print("Completed OPD Appointments")
print("-" * 80)
display(df_completed_appointments)

Completed OPD Appointments
--------------------------------------------------------------------------------


,APPOINTMENT_ID,PATIENT_NAME,DOCTOR_NAME,DEPARTMENT_NAME,APPOINTMENT_DATE,BILL_AMOUNT,PAYMENT_STATUS
0,1001,Rahul,Dr. Sharma,Cardiology,2026-07-01,800,Paid
1,1002,Sneha,Dr. Reddy,Orthopedics,2026-07-01,700,Paid
2,1005,Kiran,Dr. Rao,Pediatrics,2026-07-02,600,Paid
3,1006,Anjali Verma,Dr. Gupta,Neurology,2026-07-03,1000,Paid
4,1007,Rakesh Patel,Dr. Thomas,Dermatology,2026-07-03,650,Paid
5,1009,Suresh Yadav,Dr. Srinivas,General Medicine,2026-07-04,550,Paid
6,1011,Arjun Das,Dr. Mehta,General Medicine,2026-07-04,500,Paid
7,1012,Divya Iyer,Dr. Rao,Pediatrics,2026-07-05,600,Paid
8,1014,Sneha,Dr. Thomas,Dermatology,2026-07-05,650,Paid
9,1015,Amit,Dr. Kapoor,Cardiology,2026-07-06,900,Paid


### Calculate Total OPD Revenue

Calculates the total revenue from all paid OPD appointments.

In [51]:
query = """
SELECT
    SUM(BILL_AMOUNT) AS TOTAL_REVENUE
FROM V_OPD_APPOINTMENT_ANALYTICS
WHERE PAYMENT_STATUS = 'Paid';
"""

cursor.execute(query)

result = cursor.fetchone()

print("Total OPD Revenue:", result[0])

Total OPD Revenue: 6950


### Department-wise Revenue Analysis

Aggregates revenue by department to identify top-performing specialties.

In [52]:
import pandas as pd

query = """
SELECT
    DEPARTMENT_NAME,
    SUM(BILL_AMOUNT) AS TOTAL_REVENUE
FROM V_OPD_APPOINTMENT_ANALYTICS
WHERE PAYMENT_STATUS = 'Paid'
GROUP BY DEPARTMENT_NAME;
"""

cursor.execute(query)

rows = cursor.fetchall()
columns = [col[0] for col in cursor.description]
df_department_revenue = pd.DataFrame(rows, columns=columns)

print("Department-wise Revenue")
print("-" * 50)
display(df_department_revenue)

Department-wise Revenue
--------------------------------------------------


,DEPARTMENT_NAME,TOTAL_REVENUE
0,Cardiology,1700
1,Orthopedics,700
2,General Medicine,1050
3,Pediatrics,1200
4,Neurology,1000
5,Dermatology,1300


### Doctor-wise Performance and Revenue

Analyzes total appointments and revenue generated by each doctor.

In [53]:
import pandas as pd

query = """
SELECT
    DOCTOR_NAME,
    DEPARTMENT_NAME,
    COUNT(*) AS TOTAL_APPOINTMENTS,
    SUM(BILL_AMOUNT) AS TOTAL_REVENUE
FROM V_OPD_APPOINTMENT_ANALYTICS
WHERE PAYMENT_STATUS = 'Paid'
GROUP BY DOCTOR_NAME, DEPARTMENT_NAME;
"""

cursor.execute(query)

rows = cursor.fetchall()
columns = [col[0] for col in cursor.description]
df_doctor_revenue = pd.DataFrame(rows, columns=columns)

print("Doctor-wise Revenue")
print("-" * 60)
display(df_doctor_revenue)

Doctor-wise Revenue
------------------------------------------------------------


,DOCTOR_NAME,DEPARTMENT_NAME,TOTAL_APPOINTMENTS,TOTAL_REVENUE
0,Dr. Sharma,Cardiology,1,800
1,Dr. Kapoor,Cardiology,1,900
2,Dr. Reddy,Orthopedics,1,700
3,Dr. Mehta,General Medicine,1,500
4,Dr. Srinivas,General Medicine,1,550
5,Dr. Rao,Pediatrics,2,1200
6,Dr. Gupta,Neurology,1,1000
7,Dr. Thomas,Dermatology,2,1300


### List Pending Appointments

Retrieves details of all appointments that are still in 'Pending' status.

In [54]:
import pandas as pd

query = """
SELECT
    APPOINTMENT_ID,
    PATIENT_NAME,
    DOCTOR_NAME,
    DEPARTMENT_NAME,
    APPOINTMENT_DATE,
    APPOINTMENT_TIME
FROM V_OPD_APPOINTMENT_ANALYTICS
WHERE APPOINTMENT_STATUS = 'Pending';
"""

cursor.execute(query)

rows = cursor.fetchall()
columns = [col[0] for col in cursor.description]
df_pending_appointments = pd.DataFrame(rows, columns=columns)

print("Pending Appointments")
print("-" * 70)
display(df_pending_appointments)

Pending Appointments
----------------------------------------------------------------------


,APPOINTMENT_ID,PATIENT_NAME,DOCTOR_NAME,DEPARTMENT_NAME,APPOINTMENT_DATE,APPOINTMENT_TIME
0,1003,Amit,Dr. Mehta,General Medicine,2026-07-02,09:30:00
1,1008,Meena Gupta,Dr. Kapoor,Cardiology,2026-07-03,11:30:00
2,1013,Rahul,Dr. Gupta,Neurology,2026-07-05,13:00:00


### Identify Unpaid Bills

Lists all billing records with 'Unpaid' status.

In [55]:
import pandas as pd

query = """
SELECT
    PATIENT_NAME,
    DOCTOR_NAME,
    DEPARTMENT_NAME,
    BILL_AMOUNT,
    PAYMENT_STATUS
FROM V_OPD_APPOINTMENT_ANALYTICS
WHERE PAYMENT_STATUS = 'Unpaid';
"""

cursor.execute(query)

rows = cursor.fetchall()
columns = [col[0] for col in cursor.description]
df_unpaid_bills = pd.DataFrame(rows, columns=columns)

print("Unpaid Bills")
print("-" * 60)
display(df_unpaid_bills)

Unpaid Bills
------------------------------------------------------------


,PATIENT_NAME,DOCTOR_NAME,DEPARTMENT_NAME,BILL_AMOUNT,PAYMENT_STATUS
0,Amit,Dr. Mehta,General Medicine,500,Unpaid
1,Meena Gupta,Dr. Kapoor,Cardiology,900,Unpaid
2,Rahul,Dr. Gupta,Neurology,1000,Unpaid


### City-wise Patient Visits

Counts the total appointments originating from each city to understand patient demographics by location.

In [56]:
import pandas as pd

query = """
SELECT
    PATIENT_CITY,
    COUNT(*) AS TOTAL_APPOINTMENTS
FROM V_OPD_APPOINTMENT_ANALYTICS
GROUP BY PATIENT_CITY;
"""

cursor.execute(query)

rows = cursor.fetchall()
columns = [col[0] for col in cursor.description]
df_city_visits = pd.DataFrame(rows, columns=columns)

print("City-wise Patient Visits")
print("-" * 60)
display(df_city_visits)

City-wise Patient Visits
------------------------------------------------------------


,PATIENT_CITY,TOTAL_APPOINTMENTS
0,Hyderabad,3
1,Vijayawada,2
2,Chennai,3
3,Bangalore,1
4,Mumbai,1
5,Pune,1
6,Delhi,1
7,Visakhapatnam,1
8,Kochi,1
9,Bhubaneswar,1


### Create `V_DEPARTMENT_DAILY_REVENUE` View

Creates a view that aggregates daily revenue and appointment counts per department for completed and paid appointments.

In [57]:
cursor.execute("""

CREATE VIEW V_DEPARTMENT_DAILY_REVENUE AS

SELECT

A.APPOINTMENT_DATE,

DEP.DEPARTMENT_NAME,

COUNT(*) AS TOTAL_APPOINTMENTS,

COUNT(B.BILL_ID) AS TOTAL_PAID_BILLS,

SUM(B.BILL_AMOUNT) AS TOTAL_REVENUE

FROM APPOINTMENTS A

JOIN DOCTORS D

ON A.DOCTOR_ID=D.DOCTOR_ID

JOIN DEPARTMENTS DEP

ON D.DEPARTMENT_ID=DEP.DEPARTMENT_ID

JOIN BILLING B

ON A.APPOINTMENT_ID=B.APPOINTMENT_ID

WHERE

A.APPOINTMENT_STATUS='Completed'

AND

B.PAYMENT_STATUS='Paid'

GROUP BY

A.APPOINTMENT_DATE,

DEP.DEPARTMENT_NAME

""")

ProgrammingError: (322, 'cannot use duplicate view name: V_DEPARTMENT_DAILY_REVENUE: line 3 col 13 (at pos 14)')

### Display Department Daily Revenue

Shows the contents of the `V_DEPARTMENT_DAILY_REVENUE` view, presenting daily revenue and appointment metrics by department.

In [58]:
import pandas as pd

query = """
SELECT *
FROM V_DEPARTMENT_DAILY_REVENUE;
"""

cursor.execute(query)

rows = cursor.fetchall()
columns = [col[0] for col in cursor.description]
df_department_daily_revenue = pd.DataFrame(rows, columns=columns)

print("Department Daily Revenue")
print("-" * 80)
display(df_department_daily_revenue)

Department Daily Revenue
--------------------------------------------------------------------------------


,APPOINTMENT_DATE,DEPARTMENT_NAME,TOTAL_APPOINTMENTS,TOTAL_PAID_BILLS,TOTAL_REVENUE
0,2026-07-01,Cardiology,1,1,800
1,2026-07-06,Cardiology,1,1,900
2,2026-07-01,Orthopedics,1,1,700
3,2026-07-04,General Medicine,2,2,1050
4,2026-07-02,Pediatrics,1,1,600
5,2026-07-05,Pediatrics,1,1,600
6,2026-07-03,Neurology,1,1,1000
7,2026-07-03,Dermatology,1,1,650
8,2026-07-05,Dermatology,1,1,650


## Comprehensive OPD Appointment Analytics

Retrieves all data from the `V_OPD_APPOINTMENT_ANALYTICS` view and displays it as a pandas DataFrame for further analysis.

In [59]:
import pandas as pd

query = """
SELECT *
FROM V_OPD_APPOINTMENT_ANALYTICS;
"""

cursor.execute(query)

rows = cursor.fetchall()

columns = [col[0] for col in cursor.description]

df = pd.DataFrame(rows, columns=columns)

df

,APPOINTMENT_ID,APPOINTMENT_DATE,APPOINTMENT_TIME,PATIENT_NAME,PATIENT_CITY,DOCTOR_NAME,DEPARTMENT_NAME,SPECIALIZATION,CONSULTATION_FEE,BILL_AMOUNT,PAYMENT_MODE,PAYMENT_STATUS,APPOINTMENT_STATUS
0,1001,2026-07-01,10:00:00,Rahul,Hyderabad,Dr. Sharma,Cardiology,Cardiologist,800,800,UPI,Paid,Completed
1,1002,2026-07-01,11:00:00,Sneha,Vijayawada,Dr. Reddy,Orthopedics,Orthopedic,700,700,Cash,Paid,Completed
2,1003,2026-07-02,09:30:00,Amit,Chennai,Dr. Mehta,General Medicine,Physician,500,500,Card,Unpaid,Pending
3,1004,2026-07-02,12:00:00,Priya,Bangalore,Dr. Sharma,Cardiology,Cardiologist,800,800,Insurance,Refunded,Cancelled
4,1005,2026-07-02,01:00:00,Kiran,Hyderabad,Dr. Rao,Pediatrics,Pediatrician,600,600,UPI,Paid,Completed
5,1006,2026-07-03,09:00:00,Anjali Verma,Mumbai,Dr. Gupta,Neurology,Neurologist,1000,1000,Card,Paid,Completed
6,1007,2026-07-03,10:15:00,Rakesh Patel,Pune,Dr. Thomas,Dermatology,Dermatologist,650,650,Cash,Paid,Completed
7,1008,2026-07-03,11:30:00,Meena Gupta,Delhi,Dr. Kapoor,Cardiology,Heart Specialist,900,900,UPI,Unpaid,Pending
8,1009,2026-07-04,09:45:00,Suresh Yadav,Visakhapatnam,Dr. Srinivas,General Medicine,General Physician,550,550,Card,Paid,Completed
9,1010,2026-07-04,10:30:00,Pooja Nair,Kochi,Dr. Reddy,Orthopedics,Orthopedic,700,700,Insurance,Refunded,Cancelled
